# 08 - Train/Test Split: making the evaluation valid

Before any model is trained, decide **what data may appear where**. A model evaluated on data that leaks from its training set produces a score that looks like performance but measures memory. This notebook explains the leak, measures it on our own data, and builds a record-level split that is deterministic, checked, and saved. **No model is trained here.**

It also applies the feature decision from the EDA: the four duplicate features are dropped and four record-relative features are added (`MODEL_FEATURES` in `src/feature_extraction.py`).

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from src.feature_extraction import MODEL_FEATURES, add_record_relative_features, load_beat_table
from src.splitting import (apply_split, build_split, check_split, load_split, patient_id, patient_stats,
                           save_split, single_source_patients)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

table = add_record_relative_features(load_beat_table())
table = table.sort_values(["record", "r_peak_sample"]).reset_index(drop=True)
print(f"{len(table)} beats from {table.record.nunique()} recordings; {len(MODEL_FEATURES)} model features:")
print(MODEL_FEATURES)

## 1. Beats, recordings, patients

- A **beat** is one row of our table: a 600 ms window around one R peak. Beats from one recording are not independent samples - they are consecutive slices of one continuous signal.
- A **recording** is one continuous ECG file. Ours are ~30 minutes long and contribute between 1,593 and 3,313 beats each. One recording is one electrode placement, one signal-quality history, one physiological state.
- A **patient** is a person. One patient can have several recordings - and then the recordings are not independent of each other either.

This happens in MIT-BIH: it has 48 recordings from 47 subjects. In our data the header of record 202 reads *"This record was taken from the same analog tape as record 201"*, and both list the same age, sex and medications (68 M, Digoxin, Hydrochlorthiazide, Inderal, KCl). Record 201 is not in our pool (the detector gate excluded it), so no patient appears twice today - but the split code groups by **patient**, not recording, so it stays correct if 201 is ever added.

The unit that must never straddle the train/test boundary is therefore the **patient**; the recording is the practical stand-in for it.

## 2. Why a random split of beats leaks

If we shuffled all 40,940 beats and put 70% in train and 30% in test, then:

1. **Neighbouring beats are near-duplicates.** Beats a second apart share the patient, the electrode contact, the noise, the morphology and the rhythm. Almost every test beat has a near-identical twin in the training set.
2. **Neighbouring beats literally share data.** A window covers 600 ms; whenever the next R peak is under 600 ms away, two beats' windows contain the same raw samples. And `rr_pre_s`/`rr_post_s` of one beat *are* the timings of its neighbours.
3. **The model can learn the patient instead of the disease.** The EDA showed that amplitude features carry patient identity (pooled AUC 0.33, but 0.46 within a record), and that abnormal share differs enormously per patient (1.8% to 77.8%). With beats of every patient on both sides, "recognise the patient, predict their base rate" is a winning strategy - and it is useless on a new patient.
4. **Per-record statistics cross the boundary.** The record-relative features are computed from a record's own beats; a random split puts some of those beats in test while their scale was fitted with the rest.

The question we actually care about is: *how well does the model do on a patient it has never seen?* Only a split in which entire patients are held out can answer that.

### Measuring it (no model needed)

For every test beat, how close is its nearest training beat in feature space (features standardised with training statistics only)? And how often does a test beat have a neighbour in the training set - or even share raw samples with one? Same training/test sizes in both designs; the record-level test set is the one built in section 3.

In [ ]:
split = build_split(table)          # section 3 explains how; needed here for the "record-level" comparison
data = apply_split(table, split)

X = data[MODEL_FEATURES].to_numpy()
is_test_record = (data.split == "test").to_numpy()

rng = np.random.default_rng(42)
is_test_random = np.zeros(len(data), dtype=bool)
is_test_random[rng.choice(len(data), size=is_test_record.sum(), replace=False)] = True


def nearest_train_distance(is_test):
    scaler = StandardScaler().fit(X[~is_test])                 # statistics from the training side only
    nn = NearestNeighbors(n_neighbors=1).fit(scaler.transform(X[~is_test]))
    return nn.kneighbors(scaler.transform(X[is_test]))[0][:, 0]


def adjacency(is_test, window=216):
    # window = 72 + 144 samples: two beats closer than this share raw signal samples
    rec, pos = data.record.to_numpy(), data.r_peak_sample.to_numpy()
    same_prev = np.r_[False, rec[1:] == rec[:-1]]
    same_next = np.r_[rec[:-1] == rec[1:], False]
    gap_prev = np.r_[np.inf, np.diff(pos)]
    gap_next = np.r_[np.diff(pos), np.inf]
    prev_in_train = np.r_[False, ~is_test[:-1]] & same_prev
    next_in_train = np.r_[~is_test[1:], False] & same_next
    neighbour = is_test & (prev_in_train | next_in_train)
    sharing = is_test & ((prev_in_train & (gap_prev < window)) | (next_in_train & (gap_next < window)))
    return 100 * neighbour.sum() / is_test.sum(), 100 * sharing.sum() / is_test.sum()


dist_random = nearest_train_distance(is_test_random)
dist_record = nearest_train_distance(is_test_record)
adj_random, adj_record = adjacency(is_test_random), adjacency(is_test_record)

leak = pd.DataFrame({
    "random beat-level split": [np.median(dist_random), np.percentile(dist_random, 90), adj_random[0], adj_random[1]],
    "record-level split": [np.median(dist_record), np.percentile(dist_record, 90), adj_record[0], adj_record[1]],
}, index=["median distance to nearest training beat", "90th-percentile distance",
          "% of test beats with an adjacent beat in train", "% of test beats sharing raw samples with a train beat"])
leak.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for dist, label, color in [(dist_random, "random beat-level split", "tab:red"),
                           (dist_record, "record-level split", "tab:blue")]:
    x = np.sort(np.maximum(dist, 1e-3))
    ax.plot(x, np.arange(1, len(x) + 1) / len(x), label=label, color=color, linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Distance to the nearest training beat (standardised features, log scale)")
ax.set_ylabel("Fraction of test beats")
ax.set_title("How close is a test beat to something the model has already seen?")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig("../results/figures/17_split_leakage_distance.png", dpi=120)
plt.show()

The leak, measured. Test beats from the record-level split are about **five times farther** from anything in the training set (median distance 1.14 vs 0.23), and the curves do not overlap even in the tail (90th percentile 1.95 vs 0.56). In the random beat-level split, **89.8% of test beats have an adjacent beat in the training set, and 29.6% share raw signal samples with one** - the model would be tested on data it has, in part, literally already seen. In the record-level split both numbers are exactly 0.

So a score from a random beat split would describe how well a model recognises *familiar patients*, which is not the question. (Once models exist, we can repeat this as a direct demonstration: the same model, scored under both splits.)

## 3. The record-level split

**Principles**

1. **Whole patients only.** Every beat of a patient goes to one side; nothing is shared.
2. **Decided from metadata only** - counts of beats and beat types per patient - never from model results, so the split cannot be tuned toward a flattering score.
3. **Deterministic.** An exhaustive search with a fixed tie-break, so there is no random seed to lose: the same table always gives the same split, and the result is saved to `results/metrics/record_split.json`.
4. **A held-out test set plus cross-validation inside the rest.** Tuning and model comparison need a validation signal; using the test set for that would leak it. So the non-test patients (the *development* set) get grouped cross-validation folds of their own, and the test set is used once at the end.

**Rules used** (all in `src/splitting.py`)

- Test = 5 patients, chosen so that the test share of *beats, abnormal beats, `V` beats and `A` beats* is as close to 30% as possible.
- No single patient may supply more than 40% of the abnormal beats on either side (otherwise a score describes one person).
- **A morphology that lives in one patient can be learned or tested, not both.** Any beat type with at least 100 beats where one patient supplies at least 80% of them keeps that patient in training. Without this rule the best-balanced test set contained record 214 (the only left-bundle-branch-block patient, 1,985 Normal beats) and record 213 (92% of all fusion beats): the model could never have learned that LBBB is Normal, and the test score would have measured extrapolation to a shape it had no way to know.
- Within the development set the same idea applies to cross-validation: the patients that are the sole source of a morphology are spread across *different* folds, so no fold is the only one that has to extrapolate.

In [ ]:
print("Forced into training (sole source of a beat type):", split["forced_to_train"])
print("Spread over different CV folds:                    ", split["cv_spread_across_folds"])
print()
print("TEST patients:       ", split["test_patients"])
print("Development patients:", split["dev_patients"])
for i, fold in enumerate(split["cv_folds"]):
    print(f"  CV fold {i}: {fold}")

In [ ]:
comp = pd.crosstab(data.patient, data.symbol.astype(str))
per_patient = pd.DataFrame({
    "split": data.groupby("patient").split.first(),
    "cv_fold": data.groupby("patient").cv_fold.first(),
    "beats": data.groupby("patient").size(),
    "abnormal": data.groupby("patient").is_abnormal.sum(),
}).join(comp[[c for c in ["V", "A", "F", "J", "a"] if c in comp]])
per_patient["abnormal_pct"] = (100 * per_patient.abnormal / per_patient.beats).round(1)
per_patient.sort_values(["split", "cv_fold", "abnormal"], ascending=[False, True, False])

In [ ]:
group = np.where(data.split == "test", "test", "dev fold " + data.cv_fold.astype(str))
by_group = data.assign(group=group)
summary = pd.DataFrame({
    "patients": by_group.groupby("group").patient.nunique(),
    "beats": by_group.groupby("group").size(),
    "abnormal": by_group.groupby("group").is_abnormal.sum(),
}).join(pd.crosstab(by_group.group, by_group.symbol.astype(str))[["V", "A", "F"]])
summary["abnormal_pct"] = (100 * summary.abnormal / summary.beats).round(1)
summary.loc["development (all folds)"] = summary.drop("test").sum()
summary.loc["development (all folds)", "abnormal_pct"] = round(
    100 * summary.loc["development (all folds)", "abnormal"] / summary.loc["development (all folds)", "beats"], 1)
summary

In [ ]:
per = per_patient.copy()
per["group"] = np.where(per.split == "test", "test", "dev fold " + per.cv_fold.astype(str))
order = per.sort_values(["group", "abnormal"], ascending=[True, False]).index
palette = {"test": "tab:red", "dev fold 0": "tab:blue", "dev fold 1": "tab:green",
           "dev fold 2": "tab:orange", "dev fold 3": "tab:purple"}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.6), gridspec_kw={"width_ratios": [1.25, 1]})

ax = axes[0]
ax.barh(order[::-1], per.loc[order[::-1], "beats"], color=[palette[g] for g in per.loc[order[::-1], "group"]])
for i, rec in enumerate(order[::-1]):
    ax.text(per.loc[rec, "beats"] + 30, i, f"{per.loc[rec, 'abnormal_pct']:.0f}% abn", va="center", fontsize=8)
ax.set_title("Every recording belongs to exactly one group")
ax.set_xlabel("Number of beats")
ax.set_ylabel("Record / patient")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in palette.values()], labels=list(palette),
          loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=5, fontsize=8)

ax = axes[1]
abn = pd.crosstab(by_group[by_group.is_abnormal == 1].group, by_group[by_group.is_abnormal == 1].symbol.astype(str))
abn = abn.reindex(list(palette))
bottom = np.zeros(len(abn))
for symbol, color in [("V", "tab:red"), ("A", "tab:blue"), ("F", "tab:green"), ("J", "tab:orange"), ("a", "tab:gray")]:
    if symbol in abn:
        ax.bar(abn.index, abn[symbol], bottom=bottom, label=symbol, color=color)
        bottom += abn[symbol].to_numpy()
ax.set_title("Abnormal beats by type in each group")
ax.set_ylabel("Number of abnormal beats")
ax.tick_params(axis="x", rotation=30)
ax.legend(title="symbol")

fig.tight_layout()
fig.savefig("../results/figures/18_record_split.png", dpi=120, bbox_inches="tight")
plt.show()

Reading the design:

- **Test set:** 5 patients, 12,956 beats (31.6%), 1,204 abnormal (9.3%). `V` comes from several patients (119, 215, 116) and `A` from two (209, 118), so neither abnormal type depends on one person. Its abnormal share (9.3%) is lower than the development set's (15.5%): the abnormal-heavy records (232, 200, 233, 213) had to stay in development to respect the 40%-per-patient limit. That is a consequence of the rules, not something to "fix".
- **Development set:** 12 patients in four folds of three, each with 6-8k beats and 10-25% abnormal. **Fold 1 is the atrial fold:** it holds 1,367 of the development set's 1,574 `A` beats, because record 232 supplies almost all of them. When fold 1 is the validation fold the model trains with hardly any atrial beats, so cross-validation can barely validate atrial detection - atrial performance will have to be judged mainly on the test set's two atrial patients. Folds 0 and 2 each hold one of the single-source morphologies (fusion in 213, LBBB in 214), spread out on purpose.
- **Coverage this evaluation will not have:** `L` (LBBB) and `F` (fusion) beats exist only in development, so we **cannot measure how the model treats them in new patients**. That is a limitation to state, not to hide.

## 4. Checks and reproducibility

`apply_split` already ran `check_split`, which fails loudly if any patient, recording or beat could sit on both sides. Spelled out, plus a round-trip through the saved file:

In [ ]:
train_records = sorted(data[data.split == "train"].record.unique())
test_records = sorted(data[data.split == "test"].record.unique())
print("train recordings:", train_records)
print("test recordings: ", test_records)
print("recordings in both:", sorted(set(train_records) & set(test_records)))
print("patients in both:  ", sorted(set(data[data.split == "train"].patient) & set(data[data.split == "test"].patient)))
print("test beats assigned to a CV fold:", int((data[data.split == "test"].cv_fold >= 0).sum()))
print("beats with no split:", int((~data.split.isin(["train", "test"])).sum()))
check_split(data)

save_split(split)
assert load_split() == build_split(table), "the saved split does not match a fresh rebuild"
print("\nSaved to results/metrics/record_split.json; rebuilding from the table gives exactly the same split.")

## 5. What data may appear where

| | **Train** (development patients) | **CV validation fold** (inside development) | **Test** (5 held-out patients) |
|---|---|---|---|
| Beats used to fit a model | yes | no - the fold's patients are left out of that fit | **never** |
| Fit scalers, imputers, clipping bounds, feature selection | yes - refit inside every CV fold, on that fold's training patients only | no | **never** |
| Choose hyperparameters / decision threshold / model | scored on the CV validation folds | scoring only | **never** - test is not used to choose anything |
| Record-relative features | computed from each record's own beats, so allowed for every record | same | same - they never mix records |
| Labels | yes | to score the fold | **only to compute the final scores, once** |
| `symbol`, `record`, `patient`, `ann_offset_ms`, `time_s`, `r_peak_sample` | never a feature (annotation-derived or identity) - error analysis only | same | same |

The test set is evaluated **once**, after the features, model, hyperparameters and threshold are frozen. If a result on the test set makes us change something, the test set is no longer a test set.

**What we have already seen - stated honestly.** The EDA (notebook 07) looked at all 17 recordings, including the ones now held out. It only informed coarse decisions (drop exact duplicates; add record-relative features), and duplicates are duplicates in any subset - but strictly, the test patients are not blind to those design choices. As a check, the evidence behind the feature changes is recomputed on the **development patients only**:

In [ ]:
dev = data[data.split == "train"]


def pooled_and_within(d, col):
    pooled = roc_auc_score(d.is_abnormal, d[col])
    per_rec = [roc_auc_score(g.is_abnormal, g[col]) for _, g in d.groupby("record") if 0 < g.is_abnormal.sum() < len(g)]
    return round(pooled, 3), round(float(np.mean(per_rec)), 3)


rows = []
for col in ["rr_pre_s", "rr_pre_rel", "rr_post_s", "rr_post_rel", "qrs_p2p_mv", "qrs_p2p_rel", "amp_std", "amp_std_rel",
            "r_amplitude_mv", "qrs_fwhm_ms"]:
    rows.append((col, *pooled_and_within(dev, col)))
pd.DataFrame(rows, columns=["feature (development patients only)", "pooled_auc", "mean_within_record_auc"])

The evidence behind the feature changes, recomputed without the held-out patients:

- **Timing is still the strongest signal:** `rr_pre_s` has a pooled AUC of 0.203 and a within-record AUC of 0.099 (abnormal beats arrive earlier).
- **The record-relative versions improve the cross-patient (pooled) comparison** - the one a model applied to new patients needs: `qrs_p2p_mv` 0.505 -> `qrs_p2p_rel` 0.746, `amp_std` 0.611 -> `amp_std_rel` 0.754, `rr_post_s` 0.601 -> `rr_post_rel` 0.737. `rr_pre_rel` changes little here (0.195 vs 0.203). The within-record column is identical for each pair by construction.
- **A correction to my own EDA wording.** Notebook 07 said the amplitude features carry "almost nothing" within a record (mean AUC 0.46). On the development patients alone, `r_amplitude_mv` has a within-record AUC of **0.291** - a strong effect - against 0.464 over all 17 records. The average over records was hiding effects of *opposite sign*: the relationship between amplitude and abnormality points one way in some patients and the other way in others (the "11 of 17 records agree" column in the EDA was the warning). So the amplitude features are not empty - they are **inconsistent across patients**, which is worse for a model that has to generalise. Notebook 07's text has been corrected.

We do **not** change the split or the feature set in response. Doing that on the strength of how the held-out patients behave would be tuning on the test set. It is recorded here as a known risk to look for in the error analysis.

## Limits of this evaluation (to carry into the README)

- **Only 5 held-out patients.** Any test score has wide uncertainty. Report per-record results next to any pooled number, and keep the number of headline metrics small.
- **Some beat types can be learned but not tested:** LBBB and fusion beats exist only in development; atrial beats are learned mainly from one patient (232) and tested on two (209, 118).
- **Selection bias from notebook 06:** the pool contains records our detector handles well, so this evaluates "new patients whose beats the detector finds", not the whole database.
- **17 recordings from 17 patients is small** for strong claims about generalising to unseen patients.
- **The split is fixed.** If test results disappoint, the split is not redrawn and the features are not re-selected to improve them.